In [86]:
import json
import csv
import re
import math

import pyasn
import pandas as pd

In [3]:
asndb = pyasn.pyasn("pyasn.2026-02-18.dat.gz")

In [4]:
# QUIC Initial fixed bits and mask
initial_header = 0b11000000
initial_header_mask = 0b11000000

# QUIC Version Negotiation fixed bits and mask
vn_header = 0b10000000
vn_header_mask = 0b10000000


In [5]:
initial = "initial_qscanner_1a1a1a1a.pkt"
initial_file = open(initial, "rb")
global sent
sent = bytearray(initial_file.read())
initial_file.close()

In [6]:
# Compare two connection IDs starting at their length fields
def conn_id_match(left: bytearray, left_start: int, right: bytearray, right_start: int):
    if left_start >= len(left) or right_start >= len(right):
        return False

    # check if lengths match
    if left[left_start] != right[right_start]:
        return False

    # check each byte of ID
    for i in range(left[left_start]):
        if (left_start + i + 1) >= len(left) or (right_start + i + 1) >= len(right):
            return False
        if left[left_start + i + 1] != right[right_start + i + 1]:
            return False

    return True


In [7]:
def classify(row):
    global sent
    data = row["data"]
    saddr = row["saddr"]
    asn = asndb.lookup(saddr)[0]

    if not isinstance(data, str):
        return [saddr,  asn, "err-data", 0, ""]

    recv = bytearray.fromhex(data)

    if len(recv) < 6 or len(sent) < 6:
        return [saddr,  asn, "err-short", 0, ""]

    sent_src_id_start = 5 + sent[5] + 1
    recv_src_id_start = 5 + recv[5] + 1

    # Version Negotiation packet: match header and version
    if (vn_header & vn_header_mask) == (recv[0] & vn_header_mask) and (recv[1:5] == bytearray.fromhex("00000000")):

        # Check if src and dest connection IDs are as expected
        if conn_id_match(sent, sent_src_id_start, recv, 5) and conn_id_match(sent, 5, recv, recv_src_id_start):
            if len(recv) <= recv_src_id_start:
               return [saddr, asn, "quic-vn-err-short", 0, ""]

            # extract versions
            recv_versions_start = recv_src_id_start + recv[recv_src_id_start] + 1
            versions_len = len(recv) - recv_versions_start
            if versions_len % 4 != 0:
                return [saddr, asn, "quic-vn-err-length", 0, ""]

            versions = []
            for i in range (versions_len // 4):
                version_begin = recv_versions_start + i * 4
                version = recv[version_begin:(version_begin + 4)].hex()
                if re.search(".?a.?a.?a.?a", version):
                    continue
                elif version == "6b3343cf":
                    version = "00000002"
                versions.append(version)

            return [saddr, asn, "quic-vn", 1, " ".join(versions)]
        return [saddr, asn, "quic-vn-err-cid", 0, ""]

    # Long Header packet: match header and version
    if (initial_header & initial_header_mask) == (recv[0] & initial_header_mask) and (recv[1:5] == sent[1:5]):

        # Check if the sent src connection ID is used as dst now
        if conn_id_match(sent, sent_src_id_start, recv, 5):
            return [saddr, asn, "quic-lh", 1, recv[1:5].hex()]
        return [saddr, asn, "quic-lh-err-cid", 0, ""]

    return [saddr, asn, "err-head", 0, ""]


In [8]:
def parse(file):
    output_file = file + ".csv"
    with open(output_file, "w") as csv_file:
        writer = csv.writer(csv_file, delimiter=',')
        writer.writerow(["addr","asn","type","vn","versions"])
        with open(file) as f:
            for line in f:
                line = json.loads(line)
                row = classify(line)
                writer.writerow(row)

In [9]:
file = "vp=nl-ens/port=443/year=2026/month=02/day=18/zmap_443_20260218.jsonl"
parse(file)
output_file_nl = file + ".csv"


In [ ]:
file = "vp=au-syd/port=443/year=2026/month=02/day=18/zmap_443_20260218.jsonl"
parse(file)
output_file_au = file + ".csv"

In [28]:
file = "vp=nl-ens/port=853/year=2026/month=02/day=18/zmap_853_20260218.jsonl"
parse(file)
output_file_853 = file + ".csv"

In [12]:
df_nl = pd.read_csv(output_file_nl)
df_au = pd.read_csv(output_file_au)

In [13]:
df_nl

,addr,asn,type,vn,versions
0,104.19.128.87,13335,quic-vn,1,00000001
1,104.21.95.86,13335,quic-vn,1,00000001
2,172.67.127.163,13335,quic-vn,1,00000001
3,172.64.185.179,13335,quic-vn,1,00000001
4,104.17.51.254,13335,quic-vn,1,00000001
...,...,...,...,...,...
1080281,167.82.109.249,54113,quic-vn,1,00000001 ff00001d ff00001b
1080282,167.82.18.192,54113,quic-vn,1,00000001 ff00001d ff00001b
1080283,167.82.58.248,54113,quic-vn,1,00000001 ff00001d ff00001b
1080284,151.101.83.227,54113,quic-vn,1,00000001 ff00001d ff00001b


In [14]:
df_nl

,addr,asn,type,vn,versions
0,104.19.128.87,13335,quic-vn,1,00000001
1,104.21.95.86,13335,quic-vn,1,00000001
2,172.67.127.163,13335,quic-vn,1,00000001
3,172.64.185.179,13335,quic-vn,1,00000001
4,104.17.51.254,13335,quic-vn,1,00000001
...,...,...,...,...,...
1080281,167.82.109.249,54113,quic-vn,1,00000001 ff00001d ff00001b
1080282,167.82.18.192,54113,quic-vn,1,00000001 ff00001d ff00001b
1080283,167.82.58.248,54113,quic-vn,1,00000001 ff00001d ff00001b
1080284,151.101.83.227,54113,quic-vn,1,00000001 ff00001d ff00001b


In [15]:
df = df_nl.merge(df_au, left_on='addr', right_on='addr', how='outer',suffixes=('_nl', '_au'))

In [16]:
df[~df["asn_nl"].isna() & ~df["asn_au"].isna()]

,addr,asn_nl,type_nl,vn_nl,versions_nl,asn_au,type_au,vn_au,versions_au
0,1.0.0.0,13335.0,quic-vn,1.0,00000001,13335.0,quic-vn,1.0,00000001
1,1.0.0.1,13335.0,quic-vn,1.0,00000001,13335.0,quic-vn,1.0,00000001
2,1.0.0.10,13335.0,quic-vn,1.0,00000001,13335.0,quic-vn,1.0,00000001
3,1.0.0.100,13335.0,quic-vn,1.0,00000001,13335.0,quic-vn,1.0,00000001
4,1.0.0.101,13335.0,quic-vn,1.0,00000001,13335.0,quic-vn,1.0,00000001
...,...,...,...,...,...,...,...,...,...
1084511,99.83.243.151,16509.0,quic-vn,1.0,00000001 ff00001d ff00001e ff00001f ff000020 f...,16509.0,quic-vn,1.0,00000001 ff00001d ff00001e ff00001f ff000020 f...
1084512,99.83.243.40,16509.0,quic-vn,1.0,00000001 ff00001d ff00001e ff00001f ff000020 f...,16509.0,quic-vn,1.0,00000001 ff00001d ff00001e ff00001f ff000020 f...
1084513,99.83.246.114,16509.0,quic-vn,1.0,00000001,16509.0,quic-vn,1.0,00000001
1084514,99.83.252.121,16509.0,quic-vn,1.0,00000001 00000002,16509.0,quic-vn,1.0,00000001 00000002


In [17]:
df[df["asn_au"].isna()].groupby("asn_nl").nunique()

,addr,type_nl,vn_nl,versions_nl,asn_au,type_au,vn_au,versions_au
asn_nl,,,,,,,,
8075.0,4,1,1,1,0,0,0,0
13335.0,10,2,2,2,0,0,0,0
15169.0,55,2,2,1,0,0,0,0
16276.0,1,1,1,1,0,0,0,0
16509.0,1,1,1,1,0,0,0,0
19281.0,14,1,1,1,0,0,0,0
45513.0,1,1,1,1,0,0,0,0
49304.0,2,1,1,2,0,0,0,0
51044.0,1,1,1,1,0,0,0,0


In [18]:
df[df["asn_nl"].isna()].groupby("asn_au").nunique()

,addr,asn_nl,type_nl,vn_nl,versions_nl,type_au,vn_au,versions_au
asn_au,,,,,,,,
6939.0,1,0,0,0,0,1,1,1
8068.0,1,0,0,0,0,1,1,1
13335.0,25,0,0,0,0,2,2,1
15169.0,45,0,0,0,0,2,2,1
16276.0,3,0,0,0,0,1,1,2
16509.0,10,0,0,0,0,1,1,2
19527.0,1,0,0,0,0,1,1,1
24429.0,4,0,0,0,0,1,1,1
30114.0,1,0,0,0,0,1,1,1


In [19]:
df[df["asn_nl"].isna()]

,addr,asn_nl,type_nl,vn_nl,versions_nl,asn_au,type_au,vn_au,versions_au
3544,103.172.111.54,NaN,NaN,NaN,NaN,209242.0,quic-vn,1.0,00000001
4109,103.55.254.94,NaN,NaN,NaN,NaN,396982.0,quic-vn,1.0,00000001 ff00001d
7853,104.16.104.240,NaN,NaN,NaN,NaN,13335.0,quic-vn,1.0,00000001
18603,104.16.142.239,NaN,NaN,NaN,NaN,13335.0,quic-vn,1.0,00000001
23035,104.16.158.81,NaN,NaN,NaN,NaN,13335.0,quic-vn,1.0,00000001
...,...,...,...,...,...,...,...,...,...
1069873,76.223.48.220,NaN,NaN,NaN,NaN,16509.0,quic-vn,1.0,00000001 00000002
1074004,8.8.4.4,NaN,NaN,NaN,NaN,15169.0,quic-vn,1.0,00000001 ff00001d
1075297,83.147.29.235,NaN,NaN,NaN,NaN,203758.0,quic-vn,1.0,51303433 51303436 51303530 ff00001d 00000001 0...
1075329,84.32.84.124,NaN,NaN,NaN,NaN,47583.0,quic-vn,1.0,00000001


In [20]:
df_nl.groupby("versions").nunique().sort_values(by="addr", ascending=False)

,addr,asn,type,vn
versions,,,,
00000001,784356,118,1,1
00000001 ff00001d ff00001b,209200,1,1,1
00000001 ff00001d,71174,8,1,1
51303339 51303433 51303436 51303530 ff00001b ff00001d 00000001,12649,3,1,1
00000001 ff00001d ff00001e ff00001f ff000020 ff000021 ff000022,881,6,1,1
00000002 00000001 abcd0000 ff00001d,881,4,1,1
00000001 00000002,374,33,1,1
00000001 ff00001d ff00001e ff00001f ff000020,325,4,1,1
51303433 51303436 51303530 ff00001b ff00001d 00000001 00000002,91,7,1,1


In [24]:
df.groupby("asn_nl").nunique().sort_values(by="versions_nl", ascending=False)

,addr,type_nl,vn_nl,versions_nl,asn_au,type_au,vn_au,versions_au
asn_nl,,,,,,,,
16276.0,307,2,2,10,1,2,2,10
16509.0,12610,2,2,6,1,2,2,6
133159.0,21,1,1,5,1,1,1,5
203758.0,25,1,1,4,1,1,1,4
13335.0,654329,2,2,3,1,2,2,3
...,...,...,...,...,...,...,...,...
47242.0,1,1,1,1,1,1,1,1
401782.0,256,1,1,1,1,1,1,1
36692.0,22,1,1,0,1,1,1,0


In [26]:
df[df["asn_nl"] == 16276]

,addr,asn_nl,type_nl,vn_nl,versions_nl,asn_au,type_au,vn_au,versions_au
529868,141.94.80.103,16276.0,quic-vn,1.0,00000001,16276.0,quic-vn,1.0,00000001
529869,141.94.80.58,16276.0,quic-vn,1.0,00000001,16276.0,quic-vn,1.0,00000001
529870,141.94.80.92,16276.0,quic-vn,1.0,00000001,16276.0,quic-vn,1.0,00000001
530717,145.239.37.174,16276.0,quic-vn,1.0,00000001,16276.0,quic-vn,1.0,00000001
530718,145.239.37.234,16276.0,quic-vn,1.0,00000001,16276.0,quic-vn,1.0,00000001
...,...,...,...,...,...,...,...,...,...
1079440,91.134.125.123,16276.0,quic-vn,1.0,00000001,16276.0,quic-vn,1.0,00000001
1079441,91.134.125.137,16276.0,quic-vn,1.0,51303433 51303436 51303530 ff00001b ff00001d 0...,16276.0,quic-vn,1.0,51303433 51303436 51303530 ff00001b ff00001d 0...
1079442,91.134.125.138,16276.0,quic-vn,1.0,51303433 51303436 51303530 ff00001b ff00001d 0...,16276.0,quic-vn,1.0,51303433 51303436 51303530 ff00001b ff00001d 0...
1079443,91.134.125.158,16276.0,quic-vn,1.0,00000001,16276.0,quic-vn,1.0,00000001


In [92]:
def name_version(v):
    if isinstance(v, float) and math.isnan(v):
        return "Unknown/" + str(v)
    v = str(v)
    if v == "00000001":
        return "v1"
    elif v == "00000002":
        return "v2"
    elif v.startswith("51"):
        return "Google's QUIC version"
    else:
        return "Other draft versions"

In [33]:
df_nl.head(5)

,addr,asn,type,vn,versions
0,104.19.128.87,13335,quic-vn,1,00000001
1,104.21.95.86,13335,quic-vn,1,00000001
2,172.67.127.163,13335,quic-vn,1,00000001
3,172.64.185.179,13335,quic-vn,1,00000001
4,104.17.51.254,13335,quic-vn,1,00000001


In [93]:
df_nl_cp = df_nl.copy()

df_nl_cp["prefix"] = df_nl_cp["addr"].str.rsplit(".", n=1).str[0] + ".0/24"

df_nl_cp["version"] = df_nl_cp["versions"].str.split(" ")
df_nl_cp = df_nl_cp.explode("version").reset_index(drop=True)
df_nl_cp["named_version"] = df_nl_cp["version"].apply(name_version)

result = (
    df_nl_cp.groupby("named_version")
         .agg(count=("prefix", "nunique"))
         .sort_values("count", ascending=False)
)

# Denominator 1: sum of per-version counts (double-counts prefixes with multiple versions)
total_versions = result["count"].sum()
result["percentage_all_versions"] = (result["count"] / total_versions * 100).round(2)

# Denominator 2: distinct prefixes in the filtered set (no double-counting)
total_prefixes = df_nl_cp["prefix"].nunique()
result["percentage_all_prefixes"] = (result["count"] / total_prefixes * 100).round(2)

display(result)

print("nr distinct versions", len(result))

print("nr prefixes (summing count)", result["count"].sum())

print("nr distinct prefixes", total_prefixes)

,count,percentage_all_versions,percentage_all_prefixes
named_version,,,
v1,5577,66.16,98.39
Other draft versions,2242,26.60,39.56
v2,362,4.29,6.39
Google's QUIC version,133,1.58,2.35
Unknown/nan,115,1.36,2.03


nr distinct versions 5
nr prefixes (summing count) 8429
nr distinct prefixes 5668


hrp prefixes

In [ ]:
pdf = pd.read_csv("hrp_prefixes.csv")

hrp_pdf = pdf[pdf["is_hrp"] == True][["prefix"]].copy()

print(hrp_pdf["prefix"].nunique())

9713


In [84]:
filtered = df_nl_cp.merge(hrp_pdf, on="prefix", how="inner")
filtered["version"] = filtered["versions"].str.split(" ")
filtered = filtered.explode("version").reset_index(drop=True)
filtered["named_version"] = filtered["version"].apply(name_version)

result = (
    filtered.groupby("named_version")
         .agg(count=("prefix", "nunique"))
         .sort_values("count", ascending=False)
)

# Denominator 1: sum of per-version counts (double-counts prefixes with multiple versions)
total_versions = result["count"].sum()
result["percentage_all_versions"] = (result["count"] / total_versions * 100).round(2)

# Denominator 2: distinct prefixes in the filtered set (no double-counting)
total_prefixes = filtered["prefix"].nunique()
result["percentage_all_prefixes"] = (result["count"] / total_prefixes * 100).round(2)

display(result)

print("nr distinct versions", len(result))

,count,percentage_all_versions,percentage_all_prefixes
named_version,,,
v1,4934,60.19,98.52
ff00001d,1772,21.62,35.38
ff00001b,869,10.60,17.35
v2,95,1.16,1.90
Unknown/nan,87,1.06,1.74
ff00001e,77,0.94,1.54
ff00001f,77,0.94,1.54
ff000020,77,0.94,1.54
ff000021,77,0.94,1.54


nr distinct versions 12


non-hrp prefixes

In [ ]:
pdf = pd.read_csv("hrp_prefixes.csv")

nonhrp_pdf = pdf[pdf["is_hrp"] == False][["prefix"]].copy()

print(nonhrp_pdf["prefix"].nunique())

4073


In [85]:
filtered = df_nl_cp.merge(nonhrp_pdf, on="prefix", how="inner")
filtered["version"] = filtered["versions"].str.split(" ")
filtered = filtered.explode("version").reset_index(drop=True)

filtered["named_version"] = filtered["version"].apply(name_version)

result = (
    filtered.groupby("named_version")
         .agg(count=("prefix", "nunique"))
         .sort_values("count", ascending=False)
)

# Denominator 1: sum of per-version counts (double-counts prefixes with multiple versions)
total_versions = result["count"].sum()
result["percentage_all_versions"] = (result["count"] / total_versions * 100).round(2)

# Denominator 2: distinct prefixes in the filtered set (no double-counting)
total_prefixes = filtered["prefix"].nunique()
result["percentage_all_prefixes"] = (result["count"] / total_prefixes * 100).round(2)

display(result)

print("nr distinct versions", len(result))

,count,percentage_all_versions,percentage_all_prefixes
named_version,,,
v1,643,21.86,97.42
ff00001d,466,15.84,70.61
v2,267,9.08,40.45
ff00001e,266,9.04,40.30
ff00001f,266,9.04,40.30
ff000020,266,9.04,40.30
ff000022,254,8.63,38.48
ff000021,253,8.60,38.33
abcd0000,90,3.06,13.64


nr distinct versions 16


# 853

In [29]:
df_doq = pd.read_csv(output_file_853)

In [30]:
df_doq.groupby("versions").nunique().sort_values(by="addr", ascending=False)

,addr,asn,type,vn
versions,,,,
00000001,1309,8,1,1
00000001 00000002,26,9,1,1
00000001 00000002 ff00001d,1,1,1,1
00000001 709a50c4 ff00001d,1,1,1,1
00000001 ff00001d ff00001e ff00001f ff000020 ff000021 ff000022,1,1,1,1


In [31]:
df_doq.groupby("asn").nunique().sort_values(by="versions", ascending=False)

,addr,type,vn,versions
asn,,,,
40509,4,1,1,4
6939,1,1,1,1
45102,6,2,2,1
212772,10,1,1,1
210004,2,1,1,1
209854,1,1,1,1
197495,6,1,1,1
53667,1,1,1,1
51468,6,1,1,1


In [32]:
df_doq[df_doq["asn"] == 40509]

,addr,asn,type,vn,versions
3,149.248.223.133,40509,quic-vn,1,00000001 00000002 ff00001d
16,188.93.151.56,40509,quic-vn,1,00000001 00000002
730,137.66.51.72,40509,quic-vn,1,00000001 709a50c4 ff00001d
732,169.155.50.166,40509,quic-vn,1,00000001 ff00001d ff00001e ff00001f ff000020 f...


In [95]:
df_doq_cp = df_doq.copy()

df_doq_cp["prefix"] = df_doq_cp["addr"].str.rsplit(".", n=1).str[0] + ".0/24"

df_doq_cp["version"] = df_doq_cp["versions"].str.split(" ")
df_doq_cp = df_doq_cp.explode("version").reset_index(drop=True)
df_doq_cp["named_version"] = df_doq_cp["version"].apply(name_version)

result = (
    df_doq_cp.groupby("named_version")
         .agg(count=("prefix", "nunique"))
         .sort_values("count", ascending=False)
)

# Denominator 1: sum of per-version counts (double-counts prefixes with multiple versions)
total_versions = result["count"].sum()
result["percentage_all_versions"] = (result["count"] / total_versions * 100).round(2)

# Denominator 2: distinct prefixes in the filtered set (no double-counting)
total_prefixes = df_doq_cp["prefix"].nunique()
result["percentage_all_prefixes"] = (result["count"] / total_prefixes * 100).round(2)

display(result)

print("nr distinct versions", len(result))

print("nr prefixes (summing count)", result["count"].sum())

print("nr distinct prefixes", total_prefixes)

,count,percentage_all_versions,percentage_all_prefixes
named_version,,,
v1,32,62.75,96.97
v2,15,29.41,45.45
Other draft versions,3,5.88,9.09
Unknown/nan,1,1.96,3.03


nr distinct versions 4
nr prefixes (summing count) 51
nr distinct prefixes 33
